# Notebook 03: Benchmark Against Prabu et al. (2026)

**GenesisAeon Package 17** — full validation against Nature Astronomy paper.

DOI: [10.1038/s41550-026-02828-3](https://doi.org/10.1038/s41550-026-02828-3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from cygnus_jet_utac import CygnusJetUTAC
from cygnus_jet_utac.benchmark import PRABU_2026_TARGETS, run_benchmark

np.random.seed(42)

# Load reference measurements
with open('../data/prabu2026_measurements.yaml') as f:
    ref_data = yaml.safe_load(f)
print('Reference loaded:', ref_data['metadata']['title'])

In [ ]:
# Run 18-year simulation
system = CygnusJetUTAC(dt=21600.0, seed=42)
print('Running 18-year simulation for benchmark...')
results = system.run_cycle(duration_years=18.0)
print('Done.')

In [ ]:
# Run benchmark
bm = run_benchmark(system)
print(bm['report_str'])

## Benchmark Visualisation

In [ ]:
obs = bm['observables']
names = list(obs.keys())
deviations = [obs[n]['deviation_pct'] for n in names]
tolerances = [obs[n]['tolerance'] * 100 for n in names]
passed = [obs[n]['passed'] for n in names]

colors = ['green' if p else 'red' for p in passed]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(names))
bars = ax.bar(x, deviations, color=colors, alpha=0.7, edgecolor='black')
for xi, tol in zip(x, tolerances):
    ax.plot([xi-0.4, xi+0.4], [tol, tol], 'k--', lw=1.5, alpha=0.5)
    ax.plot([xi-0.4, xi+0.4], [-tol, -tol], 'k--', lw=1.5, alpha=0.5)

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=8)
ax.set_ylabel('Deviation from Prabu 2026 (%)')
ax.set_title(f'Benchmark Score: {bm["score"]:.1f}/100  ({bm["n_passed"]}/{bm["n_total"]} passed)')
ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='green', label='Pass'),
    Patch(facecolor='red', label='Fail'),
], loc='upper right')

plt.tight_layout()
plt.savefig('../docs/benchmark_chart.png', dpi=150)
plt.show()
print('Saved: docs/benchmark_chart.png')

In [ ]:
print('=== LaTeX Table ===')
print(bm['latex_table'])